# PhonNet Training Pipeline

This notebook covers the training of **PhonNet**, which processes speech inputs to detect **Stuttering** (sound repetitions, prolongations, blocks), **Cluttering**, **Word-Finding Difficulties**, and **Auditory Processing Disorder (APD)**.

## Modality & Scope
- Input: Audio wave sequences (.wav) + phonological task response latency
- Output: Disfluency class probabilities (Repetitions, Prolongations, Blocks, Fluent)
- Base Network: **IndicWav2Vec2** fine-tuning

## Target Runtime
- Google Colab (Free T4 GPU)

## Step 1: Environment Setup

In [ ]:
# Install transformers, datasets, librosa, and audio utils
!pip install -q transformers datasets soundfile librosa pylangacq accelerate

In [ ]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Wav2Vec2Model
from datasets import load_dataset

print("PyTorch version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Step 2: Dataset Acquisition
We load two primary datasets:
1. **AI4Bharat IndicSUPERB** (Hindi/Marathi) via Hugging Face - base pre-trained models representation validation
2. **SEP-28k** Stuttering dataset or mock data placeholders

In [ ]:
# Load AI4Bharat IndicSUPERB for Hindi speech validation
print("Loading IndicSUPERB dataset...")
try:
    indic_superb = load_dataset('ai4bharat/indicsuperb', 'hi', split='train', streaming=True)
    sample = next(iter(indic_superb))
    print("Loaded IndicSUPERB sample sample rate:", sample['audio']['sampling_rate'])
except Exception as e:
    print("Could not fetch IndicSUPERB automatically, skipping streaming check:", e)

# Download SEP-28k dataset files
# SEP-28k is a stuttering event dataset. Download mapping structure:
!git clone https://github.com/apple/ml-stuttering-events-dataset.git dataset/sep28k
print("SEP-28K Github configs fetched.")

## Step 3: Model Pipeline Setup (IndicWav2Vec2)
We load the `ai4bharat/indicwav2vec-hindi` pre-trained checkpoint and build a sequence classification head on top.

In [ ]:
model_name = "ai4bharat/indicwav2vec-hindi"
processor = Wav2Vec2Processor.from_pretrained(model_name)
wav2vec_base = Wav2Vec2Model.from_pretrained(model_name)

class StutteringClassifier(nn.Module):
    def __init__(self, base_model, num_classes=4):
        super(StutteringClassifier, self).__init__()
        self.base_model = base_model
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(768, 256), # Wav2Vec2 hidden dimension is 768
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    fun forward(self, input_values):
        outputs = self.base_model(input_values)
        # Hidden states: [batch_size, sequence_length, hidden_size]
        hidden_states = outputs.last_hidden_state
        # Transpose for pooling: [batch_size, hidden_size, sequence_length]
        hidden_states = hidden_states.transpose(1, 2)
        pooled = self.pooling(hidden_states).squeeze(-1)
        logits = self.fc(pooled)
        return logits

phonnet_model = StutteringClassifier(wav2vec_base).to(device)
print(phonnet_model)

## Step 4: Training & Fine-Tuning
We train on raw 16kHz speech. Preprocessing consists of padding/truncating inputs to 3-second segments.

In [ ]:
# Dry-run mock dataset training loader
def generate_dummy_audio_batch(batch_size=8, duration_sec=3, sample_rate=16000):
    inputs = torch.randn(batch_size, duration_sec * sample_rate).to(device)
    labels = torch.randint(0, 4, (batch_size,)).to(device) # 4 classes (REP, PROL, INT, FLUENT)
    return inputs, labels

# Simple validation step
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(phonnet_model.parameters(), lr=1e-4)

print("Starting training loop dry-run...")
phonnet_model.train()
for epoch in range(1):
    inputs, targets = generate_dummy_audio_batch()
    optimizer.zero_grad()
    logits = phonnet_model(inputs)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()
    print(f"Batch Loss: {loss.item():.4f}")

## Step 5: Export to TFLite (Float16 Quantized)
To serialize PyTorch models to TFLite, we first export to ONNX, then convert the ONNX structure into TensorFlow/TFLite representation.

In [ ]:
# Export PyTorch model to ONNX
dummy_input = torch.randn(1, 16000 * 3).to(device)
onnx_path = "phonnet.onnx"
torch.onnx.export(
    phonnet_model.cpu(),
    dummy_input.cpu(),
    onnx_path,
    input_names=['input_values'],
    output_names=['logits'],
    dynamic_axes={'input_values': {0: 'batch_size', 1: 'sequence_length'}, 'logits': {0: 'batch_size'}},
    opset_version=14
)
print("ONNX export completed successfully.")

# Install onnx-tf converter
!pip install -q onnx-tf
import onnx
from onnx_tf.backend import prepare

onnx_model = onnx.load(onnx_path)
tf_rep = prepare(onnx_model)
tf_rep.export_graph("phonnet_tf")
print("TensorFlow graph exported successfully.")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_saved_model("phonnet_tf")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
tflite_model = converter.convert()

output_path = "output/seren_phonnet.tflite"
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f"PhonNet TFLite model successfully written to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")